# NB HU37


In [6]:
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# librerias para procesamiento de archivos raster (TIFF), shapefiles y dataframes
%pip install numpy
%pip install pandas
%pip install geopandas
%pip install rasterio
%pip install --upgrade "pyarrow>=16.0.0"
%pip install "affine==2.4.0"

StatementMeta(, 0bf7312a-660e-49cc-a50e-b28806ea5dba, 29, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 63.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Not uninstalling pyarrow at /home/trusted-service-user/cluster-env/trident_env/lib/pyt

##### HU-37 Rasterizar a 10 × 10 metros todas las propiedades 

In [9]:
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# HU-37 Rasterizar a 10 × 10 metros todas las propiedades (data TIFF persistentes en ONELIKE FABRIC)
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# Modo simulacion para comprobar el flujo de datos de local a fabric para no generar data GIGO (Garbage In, Garbage Out
#   y no consumir memoria en generacion de rasters
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

import os
import re
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

from rasterio.transform import from_origin
from rasterio.features import rasterize

#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# MODO DE EJECUCIÓN
#   True  -> Simulación en Fabric (no lee/escribe archivos reales)
#   False -> Lógica original (pensada para entorno local)
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
MODO_SIMULACION = True

#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# CONFIGURACIÓN BÁSICA (compartida)
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
ANOS = [2026]

ATRIBUTOS = [
    "Variedad_TIFF",
    "Corte",
    "Edad",
    "Rdto",
    "TAH",
    "TAHM",
    "%Sac.Cana",
    "Num.Riegos",
    "Vejez",
    "Semanas mad.",
    "Luvias",
    "Lluvias (0 - 3 Meses)",
    "Lluvias 10d Ant.Cos",
    "Ton.Azucar",
]

RESOLUCION = 10
CRS_SALIDA = "EPSG:21896"
NODATA = -9999

print("\n")
print("=" * 80)
print("GENERADOR RÁPIDO DE RASTER HISTÓRICOS")
print("=" * 80)

print(f"\nAños seleccionados: {ANOS}")
print(f"Atributos: {ATRIBUTOS}")
print(f"Resolución: {RESOLUCION} x {RESOLUCION} metros")

if MODO_SIMULACION:
    BASE = "Files"
    CARPETA_CONSOLIDADO = f"{BASE}/CONSOLIDADO"
    CARPETA_BIOSALC = f"{CARPETA_CONSOLIDADO}/BIOSALC"
    CARPETA_CAPAS_GIS = f"{CARPETA_CONSOLIDADO}/CAPAS_GIS"
    CARPETA_SALIDA_ONELAKE = f"{CARPETA_CONSOLIDADO}/RASTER_HISTORICOS"
    RUTA_PARQUET = f"{CARPETA_BIOSALC}/df_crono.parquet"
    RUTA_SHP_STE = f"{CARPETA_CAPAS_GIS}/Ste.shp"

    print("\n" + "=" * 80)
    print("MODO SIMULACIÓN ACTIVADO")
    print("=" * 80 + "\n")
    print("Este cuaderno está ejecutándose en modo simulación dentro de Fabric.")
    print("No se leerán archivos Parquet ni Shapefiles reales y no se generarán TIFF.\n")
    print("Resumen simulado del escenario:")
    print(f"  - Años a procesar: {ANOS}")
    print(f"  - Número de atributos a rasterizar: {len(ATRIBUTOS)}")
    print(f"  - Parquet de entrada (simulado): {RUTA_PARQUET}")
    print(f"  - Shapefile de lotes (simulado): {RUTA_SHP_STE}")
    print(f"  - Carpeta de salida (simulada): {CARPETA_SALIDA_ONELAKE}\n")

    print("Simulando etapas principales:")
    print("  [OK] 1. Lectura del Parquet df_crono.parquet desde Lakehouse Files.")
    print("  [OK] 2. Normalización de CODSIA y filtrado por años seleccionados.")
    print("  [OK] 3. Identificación de shapefiles en CONSOLIDADO/CAPAS_GIS.")
    print("  [OK] 4. Unión de df_crono con geometrías de Ste.shp por CODSIA.")
    print("  [OK] 5. Cálculo de extensión espacial y malla de 10 x 10 m.")
    print("  [OK] 6. Rasterización de todos los atributos para cada año.")
    print("  [OK] 7. Escritura simulada de TIFF en CONSOLIDADO/RASTER_HISTORICOS.")
    print("  [OK] 8. Registro de control interno del proceso (simulado).\n")

    print("ESTADO GENERAL: EJECUCIÓN COMPLETADA CORRECTAMENTE (SIMULADA)\n")
    print("Para activar el procesamiento real:")
    print("  - Cambie MODO_SIMULACION = False en este cuaderno.")
    print("  - Ajuste las rutas a df_crono.parquet y Ste.shp si es necesario.")
    print("  - Verifique permisos de lectura/escritura sobre Lakehouse Files.\n")
else:
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # 1. CONFIGURACIÓN (MODO REAL / LOCAL)
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # CARPETA PRINCIPAL
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    CARPETA_DATOS = (
        r"C:\Users\jjruiz\OneDrive - Sector Agro\Geodata\Analisis datos a raster\TEMP\depurados"
    )

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # ARCHIVO EXCEL
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    ARCHIVO_EXCEL = os.path.join(
        CARPETA_DATOS,
        "Reporte_Biosal.xlsx"
    )

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # CARPETA DE SALIDA
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    CARPETA_SALIDA = os.path.join(
        CARPETA_DATOS,
        "RASTER_HISTORICOS"
    )

    os.makedirs(
        CARPETA_SALIDA,
        exist_ok=True
    )

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # 2. FUNCIONES AUXILIARES
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

    def normalizar_codsia(valor):
        if pd.isna(valor):
            return None

        texto = str(valor).strip()
        texto = texto.strip()

        if texto.endswith(".0"):
            texto = texto[:-2]

        if texto.isdigit():
            texto = texto.zfill(9)
            return texto

        match = re.fullmatch(
            r"(\d+)([A-Za-z])",
            texto
        )

        if match:
            parte_numerica = match.group(1)
            letra = match.group(2)

            parte_numerica = parte_numerica.zfill(9)

            return parte_numerica + letra

        return texto

    def limpiar_nombre_archivo(nombre):
        nombre = str(nombre).strip()
        nombre = nombre.replace(" ", "_")

        nombre = re.sub(
            r"[^A-Za-z0-9_\-]",
            "",
            nombre
        )

        return nombre.upper()

    def obtener_ruta_shp(nombre):
        nombre = str(nombre).strip()

        if not nombre.lower().endswith(".shp"):
            nombre += ".shp"

        return os.path.join(
            CARPETA_DATOS,
            nombre
        )

    def convertir_a_numerico(serie):
        if pd.api.types.is_numeric_dtype(serie):
            return pd.to_numeric(
                serie,
                errors="coerce"
            )

        serie = (
            serie
            .astype("string")
            .str.strip()
        )

        serie = serie.str.replace(
            ",",
            ".",
            regex=False
        )

        return pd.to_numeric(
            serie,
            errors="coerce"
        )

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # 3. INICIO
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

    print("\n")
    print("=" * 80)
    print("LECTURA DEL EXCEL")
    print("=" * 80)

    print("Leyendo únicamente las columnas necesarias...")

    if not os.path.exists(ARCHIVO_EXCEL):
        raise FileNotFoundError(
            f"\nNo existe el archivo:\n{ARCHIVO_EXCEL}"
        )

    COLUMNAS_EXCEL = [
        "CODSIA",
        "year",
        "Capa_producto_final"
    ] + ATRIBUTOS

    COLUMNAS_EXCEL = list(
        dict.fromkeys(
            COLUMNAS_EXCEL
        )
    )

    df = pd.read_excel(
        ARCHIVO_EXCEL,
        sheet_name="Datos_depurados_eda_quemar",
        usecols=COLUMNAS_EXCEL,
        dtype={
            "CODSIA": "string"
        }
    )

    print(
        f"Filas leídas: {len(df):,}"
    )

    COLUMNAS_REQUERIDAS = [
        "CODSIA",
        "year",
        "Capa_producto_final"
    ] + ATRIBUTOS

    faltantes = [
        columna
        for columna in COLUMNAS_REQUERIDAS
        if columna not in df.columns
    ]

    if faltantes:
        raise ValueError(
            "\nFaltan columnas en el Excel:\n\n"
            + "\n".join(faltantes)
        )

    df["CODSIA"] = (
        df["CODSIA"]
        .apply(normalizar_codsia)
        .astype("string")
    )

    df["year"] = pd.to_numeric(
        df["year"],
        errors="coerce"
    )

    df["Capa_producto_final"] = (
        df["Capa_producto_final"]
        .astype("string")
        .str.strip()
    )

    df_proceso = df[
        df["year"].isin(ANOS)
    ].copy()

    del df

    print("\n")
    print("=" * 80)
    print("DATOS SELECCIONADOS")
    print("=" * 80)

    print(f"Años seleccionados: {ANOS}")
    print(f"Registros a procesar: {len(df_proceso):,}")

    if df_proceso.empty:
        print("\nNo existen registros para los años seleccionados.")
        raise SystemExit

    for anio in ANOS:
        print("\n\n")
        print("#" * 80)
        print(f"# PROCESANDO {anio}")
        print("#" * 80)

        df_anio = df_proceso[
            df_proceso["year"] == anio
        ].copy()

        print(
            f"Registros: {len(df_anio):,}"
        )

        if df_anio.empty:
            print("No existen registros.")
            continue

        grupos = df_anio.groupby(
            "Capa_producto_final",
            dropna=False
        )

        print(
            f"SHP diferentes: {len(grupos):,}"
        )

        lista_gdf = []

        for (
            nombre_capa,
            df_capa
        ) in grupos:
            print("\n")
            print("-" * 70)
            print(f"Procesando: {nombre_capa}")
            print("-" * 70)

            if pd.isna(nombre_capa):
                print("⚠ Capa vacía")
                continue

            ruta_shp = obtener_ruta_shp(
                nombre_capa
            )

            print(ruta_shp)

            if not os.path.exists(ruta_shp):
                print("❌ SHP NO ENCONTRADO")
                continue

            try:
                gdf = gpd.read_file(
                    ruta_shp
                )
            except Exception as e:
                print(
                    f"❌ Error leyendo SHP: {e}"
                )
                continue

            print(
                f"Polígonos SHP: {len(gdf):,}"
            )

            if "CODSIA" not in gdf.columns:
                print(
                    "❌ El SHP no contiene la columna CODSIA"
                )
                continue

            gdf["CODSIA"] = (
                gdf["CODSIA"]
                .apply(normalizar_codsia)
                .astype("string")
            )

            if gdf.crs is None:
                print("❌ El SHP no tiene CRS.")
                continue

            if gdf.crs != CRS_SALIDA:
                gdf = gdf.to_crs(
                    CRS_SALIDA
                )

            codsias_excel = set(
                df_capa["CODSIA"]
                .dropna()
                .unique()
            )

            gdf = gdf[
                gdf["CODSIA"].isin(
                    codsias_excel
                )
            ].copy()

            print(
                f"Polígonos relacionados: {len(gdf):,}"
            )

            if gdf.empty:
                continue

            datos_raster = df_capa[
                ["CODSIA"] + ATRIBUTOS
            ].copy()

            registros_raster = []

            for (
                codsia,
                grupo
            ) in datos_raster.groupby(
                "CODSIA",
                dropna=False
            ):
                if pd.isna(codsia):
                    continue

                fila_raster = {
                    "CODSIA": codsia
                }

                for atributo in ATRIBUTOS:
                    valores = (
                        grupo[atributo]
                        .dropna()
                        .unique()
                    )

                    if len(valores) == 0:
                        fila_raster[atributo] = np.nan
                    else:
                        fila_raster[atributo] = valores[0]

                registros_raster.append(
                    fila_raster
                )

            datos_raster = pd.DataFrame(
                registros_raster
            )

            gdf_resultado = gdf.merge(
                datos_raster,
                on="CODSIA",
                how="inner"
            )

            print(
                f"Polígonos para raster: {len(gdf_resultado):,}"
            )

            if gdf_resultado.empty:
                continue

            gdf_resultado = (
                gdf_resultado[
                    gdf_resultado.geometry.notna()
                ].copy()
            )

            if gdf_resultado.empty:
                continue

            invalidas = (
                ~gdf_resultado.geometry.is_valid
            )

            if invalidas.any():
                print(
                    f"Reparando {invalidas.sum():,} geometrías..."
                )
                try:
                    gdf_resultado.loc[
                        invalidas,
                        "geometry"
                    ] = (
                        gdf_resultado.loc[
                            invalidas,
                            "geometry"
                        ].make_valid()
                    )
                except Exception as e:
                    print(
                        f"⚠ Advertencia: {e}"
                    )

            gdf_resultado = gdf_resultado[
                ["CODSIA", "geometry"] + ATRIBUTOS
            ].copy()

            lista_gdf.append(
                gdf_resultado
            )

        if not lista_gdf:
            print(
                f"\nNo hay geometrías para {anio}."
            )
            continue

        gdf_anio = gpd.GeoDataFrame(
            pd.concat(
                lista_gdf,
                ignore_index=True
            ),
            crs=CRS_SALIDA
        )

        print(
            f"\nPolígonos totales {anio}: {len(gdf_anio):,}"
        )

        minx, miny, maxx, maxy = (
            gdf_anio.total_bounds
        )

        minx = (
            np.floor(
                minx / RESOLUCION
            ) * RESOLUCION
        )

        miny = (
            np.floor(
                miny / RESOLUCION
            ) * RESOLUCION
        )

        maxx = (
            np.ceil(
                maxx / RESOLUCION
            ) * RESOLUCION
        )

        maxy = (
            np.ceil(
                maxy / RESOLUCION
            ) * RESOLUCION
        )

        ancho = int(
            round(
                (maxx - minx) / RESOLUCION
            )
        )

        alto = int(
            round(
                (maxy - miny) / RESOLUCION
            )
        )

        print(
            f"Dimensiones: {ancho:,} x {alto:,}"
        )

        transform = from_origin(
            minx,
            maxy,
            RESOLUCION,
            RESOLUCION
        )

        for atributo in ATRIBUTOS:
            print("\n")
            print("=" * 70)
            print(f"RASTERIZANDO: {atributo}")
            print("=" * 70)

            gdf_raster = gdf_anio[
                ["CODSIA", "geometry", atributo]
            ].copy()

            gdf_raster["_VALOR"] = convertir_a_numerico(
                gdf_raster[atributo]
            )

            gdf_raster = gdf_raster[
                gdf_raster["_VALOR"].notna()
            ].copy()

            if gdf_raster.empty:
                print(
                    "No existen valores para rasterizar."
                )
                continue

            shapes = zip(
                gdf_raster.geometry,
                gdf_raster["_VALOR"]
            )

            raster = rasterize(
                shapes=shapes,
                out_shape=(
                    alto,
                    ancho
                ),
                transform=transform,
                fill=NODATA,
                dtype="float32",
                all_touched=True
            )

            nombre_atributo = limpiar_nombre_archivo(
                atributo
            )

            nombre_tiff = (
                f"{anio}_{nombre_atributo}.tif"
            )

            ruta_tiff = os.path.join(
                CARPETA_SALIDA,
                nombre_tiff
            )

            with rasterio.open(
                ruta_tiff,
                "w",
                driver="GTiff",
                height=alto,
                width=ancho,
                count=1,
                dtype="float32",
                crs=CRS_SALIDA,
                transform=transform,
                nodata=NODATA,
                compress="lzw",
                tiled=True,
                BIGTIFF="IF_SAFER"
            ) as dst:
                dst.write(
                    raster,
                    1
                )

                dst.update_tags(
                    YEAR=str(anio),
                    ATTRIBUTE=str(atributo),
                    RESOLUTION=f"{RESOLUCION}x{RESOLUCION} m",
                    CRS=CRS_SALIDA,
                    SOURCE_EXCEL=os.path.basename(
                        ARCHIVO_EXCEL
                    )
                )

            print(
                f"✓ Guardado: {ruta_tiff}"
            )

        del gdf_anio
        del lista_gdf
        del df_anio

    print("\n")
    print("=" * 80)
    print("PROCESO TERMINADO")
    print("=" * 80)

    print(
        f"\nAños procesados: {ANOS}"
    )

    print(
        f"\nCarpeta de resultados:"
    )

    print(
        CARPETA_SALIDA
    )

    print(
        "\nÚnicamente se generaron los raster solicitados."
    )


StatementMeta(, 0bf7312a-660e-49cc-a50e-b28806ea5dba, 33, Finished, Available, Finished, False)



GENERADOR RÁPIDO DE RASTER HISTÓRICOS

Años seleccionados: [2026]
Atributos: ['Variedad_TIFF', 'Corte', 'Edad', 'Rdto', 'TAH', 'TAHM', '%Sac.Cana', 'Num.Riegos', 'Vejez', 'Semanas mad.', 'Luvias', 'Lluvias (0 - 3 Meses)', 'Lluvias 10d Ant.Cos', 'Ton.Azucar']
Resolución: 10 x 10 metros

MODO SIMULACIÓN ACTIVADO

Este cuaderno está ejecutándose en modo simulación dentro de Fabric.
No se leerán archivos Parquet ni Shapefiles reales y no se generarán TIFF.

Resumen simulado del escenario:
  - Años a procesar: [2026]
  - Número de atributos a rasterizar: 14
  - Parquet de entrada (simulado): Files/CONSOLIDADO/BIOSALC/df_crono.parquet
  - Shapefile de lotes (simulado): Files/CONSOLIDADO/CAPAS_GIS/Ste.shp
  - Carpeta de salida (simulada): Files/CONSOLIDADO/RASTER_HISTORICOS

Simulando etapas principales:
  [OK] 1. Lectura del Parquet df_crono.parquet desde Lakehouse Files.
  [OK] 2. Normalización de CODSIA y filtrado por años seleccionados.
  [OK] 3. Identificación de shapefiles en CONSOLID